In [10]:
from oqd_compiler_infrastructure import Post, PrettyPrint

from oqd_core.atomic import AtomicCFGBuilder
from oqd_core.frontend.atomic import parse_atomic

printer = Post(PrettyPrint())

with open("test.atomic", mode="r", encoding="utf8") as f:
    source = f.read()

source = """
r = ionreg(3)
q0 = r[0]
"""
circuit = parse_atomic(source)
cfg = AtomicCFGBuilder()(circuit)

In [11]:
from oqd_core.atomic import (
    AtomicBoundsChecker,
    AtomicCFGBuilder,
    AtomicTypeChecker,
    AvailableVariableAnalysis,
    ConstantFolding,
    serialize_atomic,
)

In [12]:
avail_checker = AvailableVariableAnalysis()
avail_res = avail_checker.analyze(cfg)

In [13]:
bounds_checker = AtomicBoundsChecker()
bounds_res = bounds_checker.analyze(cfg)

In [14]:
checker = AtomicTypeChecker()
res = checker.analyze(cfg)

In [15]:
from oqd_compiler_infrastructure import FixedPoint, InplacePre
from rich import print as pprint

from oqd_core.analysis.analog.fold import ConstantFoldingRule

fold_checker = ConstantFolding()
fold_res = fold_checker.analyze(cfg)
# pprint(fold_res)

fold_pass = FixedPoint(InplacePre(ConstantFoldingRule(fold_res=fold_res)))
# fold_pass = InplacePre(ConstantFoldingRule(fold_res=fold_res))
# fold_pass = ConstantFoldingRule(fold_res=fold_res)

fold_pass(cfg)


pprint(cfg)

ValidationError: 1 validation error for Extract
value
  Value error, Failed to cast value to AtomicExpr [type=value_error, input_value=<class 'oqd_core.analysis.atomic.fold.CInvalid'>, input_type=type]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error

In [ ]:
from oqd_core.compiler.atomic import compile_atomic_circuit

circuit, cfg = compile_atomic_circuit(source)

In [ ]:
circuit

AtomicCircuit(class_='AtomicCircuit', statements=[Declaration(class_='Declaration', name='r', value=IonRegister(class_='IonRegister', size=Constant(class_='Constant', value=3)))])

In [ ]:
from oqd_compiler_infrastructure import cfg_to_dot

dot = cfg_to_dot(cfg, serialize=serialize_atomic, max_lines=5)
print(dot.source)
dot.render("cfg", format="png", cleanup=True)

digraph {
	0 [label=" Block #0
------------------------
--------- tags ---------
__label__: entry"]
	0 -> 1
	1 [label=" Block #1
------------------------
r = ionreg(3)"]
	1 -> 2
	2 [label=" Block #2
------------------------
--------- tags ---------
__label__: exit"]
}



'cfg.png'